# Costruzione decrizione automatica svg funzioni

## Esempio estrazione

In [103]:
import xml.etree.ElementTree as ET
import re

def apri_file_svg(nome_file):
    tree = ET.parse(nome_file)
    root = tree.getroot()
    return root

# Esempio di utilizzo
nome_file_svg = 'funzione.svg'
root_svg = apri_file_svg(nome_file_svg)

#find element with id="funzioni1"
for child in root_svg:
    if child.attrib['id'] == 'funzioni1':
        funzioni1 = child

#foreach element in funzioni1 print id
for child in funzioni1:
    if not re.search('title', child.attrib['id']):
        print(child.attrib['id'])

        #forreach element in child print id
        for child2 in child:
            if not re.search('title', child2.attrib['id']):
                print('- ', child2.attrib['id'])
                
                #get tspan
                for child3 in child2:
                    if child3.tag == '{http://www.w3.org/2000/svg}tspan':
                        print('---', child3.text)


    print('===============================')

dominio
-  nome-dominio
--- A
-  insieme-dominio
-  josephine
--- Josephine
-  kevin
--- Kevin
-  luigi
--- Luigi
-  silvia
--- Silvia
codominio
-  nome-codominio
--- B
-  insieme-codominio
-  bari
--- Bari
-  roma
--- Roma
-  milano
--- Milano
-  bologna
--- Bologna
-  siena
--- Siena
link-josephine-bari
link-silvia-milano
link-luigi-milano
link-kevin-siena
f-silvia-milano
-  tspan1
f-josephine-bari
-  tspan1-3
f-luigi-milano
-  tspan1-8
f-kevin-siena
-  tspan1-5


## Gestione delle coppie presenti nod1-nodo2

### Estrazione

In [104]:
import xml.etree.ElementTree as ET
import re

def apri_file_svg(nome_file):
    tree = ET.parse(nome_file)
    root = tree.getroot()
    return root

# Esempio di utilizzo
nome_file_svg = 'funzione.svg'
root_svg = apri_file_svg(nome_file_svg)

#find element with id="funzioni1"
for child in root_svg:
    if child.attrib['id'] == 'funzioni1':
        funzioni1 = child

componenti = {}
for child in funzioni1:
    if not re.search('title', child.attrib['id']):
        nome_componente = child.attrib['id']
        componenti[nome_componente] = []

        componente = {}
        for child2 in child:
            if not re.search('title', child2.attrib['id']):
                componente[child2.attrib['id']] = []
               
                #get tspan
                for child3 in child2:
                    if child3.tag == '{http://www.w3.org/2000/svg}tspan':
                        componente[child2.attrib['id']] = child3.text

        componenti[nome_componente].append(componente)

componenti

{'dominio': [{'nome-dominio': 'A',
   'insieme-dominio': [],
   'josephine': 'Josephine',
   'kevin': 'Kevin',
   'luigi': 'Luigi',
   'silvia': 'Silvia'}],
 'codominio': [{'nome-codominio': 'B',
   'insieme-codominio': [],
   'bari': 'Bari',
   'roma': 'Roma',
   'milano': 'Milano',
   'bologna': 'Bologna',
   'siena': 'Siena'}],
 'link-josephine-bari': [{}],
 'link-silvia-milano': [{}],
 'link-luigi-milano': [{}],
 'link-kevin-siena': [{}],
 'f-silvia-milano': [{'tspan1': []}],
 'f-josephine-bari': [{'tspan1-3': []}],
 'f-luigi-milano': [{'tspan1-8': []}],
 'f-kevin-siena': [{'tspan1-5': []}]}

### Generazione

In [105]:
nome_insieme_dominio = ''
elementi_dominio = []

nome_insieme_codominio = ''
elementi_codominio = []

link = {}
for componente in componenti:
    if (componente == 'dominio'):
        for child in componenti[componente]:
            for child2 in child:
                if re.search('nome', child2):
                    nome_insieme_dominio = child[child2]
                elif not re.search('insieme', child2):
                    elementi_dominio.append(child2)

    elif (componente == 'codominio'):
        
        for child in componenti[componente]:
            for child2 in child:
                if re.search('nome', child2):
                    nome_insieme_codominio = child[child2]
                elif not re.search('insieme', child2):
                    elementi_codominio.append(child2)

    else: #link
        componenti_split = componente.split('-')
        link[componenti_split[1]] = componenti_split[2]

print('DOMINIO:', nome_insieme_dominio, elementi_dominio)
print('CODOMINIO:', nome_insieme_codominio, elementi_codominio)
print('LINK:', link)


DOMINIO: A ['josephine', 'kevin', 'luigi', 'silvia']
CODOMINIO: B ['bari', 'roma', 'milano', 'bologna', 'siena']
LINK: {'josephine': 'bari', 'silvia': 'milano', 'luigi': 'milano', 'kevin': 'siena'}


## Dialogue acts

I dialogue acts utilizzati sono i seguenti
- <b>DS:opening</b>: saluti iniziali (es. ciao, buongiorno, ...)
- <b>Ta:setQuestion</b>: domanda specifica (es. quante elementi nel dominio, ...)
- <b>Ta:request</b>: richiesta generica (es. parlami di, descrivimi, ...)
- <b>Ta:propositionalQuestion</b>: domanda che richiede una risposta che confermi o confuti una dichiarazione (es. è vero che ...)

Variabili:
- Dialogue act
- elemento1
- elemento2
- insieme
- topic

In [106]:
# openingAnswer = {}
setQuestionAnswer = {}
requestAnswer = {}
propositionalQuestionAnswer = {}

vars = {
    'dialogue_act': 'none',
    'elemento1': 'none',
    'elemento2': 'none',
    'insieme': 'none',
    'topic': 'none',
}


### Ta:setQuestion

Esempio: 
- Quanti elementi nel dominio?
- A cosa è associato x?
- ...

1. Caso in cui l'elemento del dominio non è collegato a nulla nel codominio

In [107]:
for elemento in elementi_dominio:
    trovato = False
    for key in link:
        if key == elemento:
            trovato = True
            break

    if not trovato:
        domande = [
            '* ' + elemento + ' * associato * ',
            '* ' + elemento + ' * collegato * ',
            '* associato * ' + elemento + ' *',
            '* collegato * ' + elemento + ' *',
            '* immagine * ' + elemento + ' *',
            '* funzione * ' + elemento + ' *',
            '* f * ' + elemento + ' *',
        ]

        # variabili
        vars = {
            'dialogue_act': 'setQuestion',
            'elemento1': elemento,
            'elemento2': 'none',
            'insieme': nome_insieme_dominio,
            'topic': 'funzione',
        }

        # contenuto testuale
        text = ''
        for var in vars:
            text += '<think><set name="' + var + '">' + vars[var] + '</set></think>'

        text += elemento + ' non è associato a nessun elemento del codominio'

        # immagine
        text += '<image>insiemi/funzione.svg</image>'
        text += '<svgElement style-name="stroke" style-value="#04ed00">' + elemento + '</svgElement>'

        for domanda in domande:
            domanda_uppercase = domanda.upper()            

            setQuestionAnswer[domanda_uppercase] = text
            print(domanda, text)

2. Caso in cui l'elemento del codominio non è collegato a nulla nel dominio

In [108]:
for elemento in elementi_codominio:
    trovato = False
    for key in link:
        if link[key] == elemento:
            trovato = True
            break

    if not trovato:
        domande = [
            '* ' + elemento + ' * associato * ',
            '* ' + elemento + ' * collegato * ',
            '* associato * ' + elemento + ' *',
            '* collegato * ' + elemento + ' *',
            '* controimmagine * ' + elemento + ' *',
            '* funzione inversa * ' + elemento + ' *',
            '* f^-1 * ' + elemento + ' *',
            '* f-1 *' + elemento + ' *',
        ]

        # variabili
        vars = {
            'dialogue_act': 'setQuestion',
            'elemento1': elemento,
            'elemento2': 'none',
            'insieme': nome_insieme_codominio,
            'topic': 'funzione_inversa',
        }

        text = ''
        for var in vars:
            text += '<think><set name="' + var + '">' + vars[var] + '</set></think>'

        # contenuto testuale
        text += elemento + ' non è associato a nessun elemento del dominio'

        # immagine
        text += '<image>insiemi/funzione.svg</image>'
        text += '<svgElement style-name="stroke" style-value="#04ed00">' + elemento + '</svgElement>'

        for domanda in domande:
            domanda_uppercase = domanda.upper()
            
            setQuestionAnswer[domanda_uppercase] =  text
            print(domanda, text)

* roma * associato *  <think><set name="dialogue_act">setQuestion</set></think><think><set name="elemento1">roma</set></think><think><set name="elemento2">none</set></think><think><set name="insieme">B</set></think><think><set name="topic">funzione_inversa</set></think>roma non è associato a nessun elemento del dominio<image>insiemi/funzione.svg</image><svgElement style-name="stroke" style-value="#04ed00">roma</svgElement>
* roma * collegato *  <think><set name="dialogue_act">setQuestion</set></think><think><set name="elemento1">roma</set></think><think><set name="elemento2">none</set></think><think><set name="insieme">B</set></think><think><set name="topic">funzione_inversa</set></think>roma non è associato a nessun elemento del dominio<image>insiemi/funzione.svg</image><svgElement style-name="stroke" style-value="#04ed00">roma</svgElement>
* associato * roma * <think><set name="dialogue_act">setQuestion</set></think><think><set name="elemento1">roma</set></think><think><set name="ele

3. Numero di elementi nel dominio

In [109]:
domande = [
    '* insieme ' + nome_insieme_dominio + ' *',
    '* dominio *',
]

# variabili
vars = {
    'dialogue_act': 'setQuestion',
    'elemento1': 'none',
    'elemento2': 'none',
    'insieme': nome_insieme_dominio,
    'topic': 'insieme',
}

text = ''
for var in vars:
    text += '<think><set name="' + var + '">' + vars[var] + '</set></think>'

# contenuto testuale
text += 'L\'insieme ' + nome_insieme_dominio + ' è composto da ' + str(len(elementi_dominio)) + ' elementi: '
for elemento in elementi_dominio:
    text += elemento + ', '
text = text[:-2] + '.'

# immagine
text += '<image>insiemi/funzione.svg</image>'
text += '<svgElement style-name="fill" style-value="#04ed00">insieme-dominio</svgElement>'

print(text)

for domanda in domande:
    domanda_uppercase = domanda.upper()
    setQuestionAnswer[domanda_uppercase] = text

<think><set name="dialogue_act">setQuestion</set></think><think><set name="elemento1">none</set></think><think><set name="elemento2">none</set></think><think><set name="insieme">A</set></think><think><set name="topic">insieme</set></think>L'insieme A è composto da 4 elementi: josephine, kevin, luigi, silvia.<image>insiemi/funzione.svg</image><svgElement style-name="fill" style-value="#04ed00">insieme-dominio</svgElement>


4. Numero di elementi nel codominio

In [110]:
domande = [
    '* insieme ' + nome_insieme_codominio + ' *',
    '* codominio *',
]

# variabili
vars = {
    'dialogue_act': 'setQuestion',
    'elemento1': 'none',
    'elemento2': 'none',
    'insieme': nome_insieme_codominio,
    'topic': 'insieme',
}

text = ''
for var in vars:
    text += '<think><set name="' + var + '">' + vars[var] + '</set></think>'

# contenuto testuale
text += 'L\'insieme ' + nome_insieme_codominio + ' è composto da ' + str(len(elementi_codominio)) + ' elementi: '
for elemento in elementi_codominio:
    text += elemento + ', '
text = text[:-2] + '.'

# immagine
text += '<image>insiemi/funzione.svg</image>'
text += '<svgElement style-name="fill" style-value="#04ed00">insieme-codominio</svgElement>'
print(text)

for domanda in domande:
    domanda_uppercase = domanda.upper()
    setQuestionAnswer[domanda_uppercase] = text

<think><set name="dialogue_act">setQuestion</set></think><think><set name="elemento1">none</set></think><think><set name="elemento2">none</set></think><think><set name="insieme">B</set></think><think><set name="topic">insieme</set></think>L'insieme B è composto da 5 elementi: bari, roma, milano, bologna, siena.<image>insiemi/funzione.svg</image><svgElement style-name="fill" style-value="#04ed00">insieme-codominio</svgElement>


5. A cosa è collegato l'elemento del dominio

In [111]:
for elemento in link:
    domande = [
        '* ' + elemento + ' *'
    ]

    # variabili
    vars = {
        'dialogue_act': 'setQuestion',
        'elemento1': elemento,
        'elemento2': 'none',
        'insieme': nome_insieme_dominio,
        'topic': 'funzione',
    }

    text = ''
    for var in vars:
        text += '<think><set name="' + var + '">' + vars[var] + '</set></think>'

    # contenuto testuale
    text += 'La funzione associa l\'elemento ' + elemento + ' dell\'insieme ' + nome_insieme_dominio + ' all\'elemento ' + link[elemento] + ' dell\'insieme ' + nome_insieme_codominio + '.'
    
    # immagine
    text += '<image>insiemi/funzione.svg</image>'
    text += '<svgElement style-name="stroke" style-value="#04ed00">link-' + elemento + '-' + link[elemento] + '</svgElement>'
    
    print(text)

    for domanda in domande:
        domanda_uppercase = domanda.upper()
        setQuestionAnswer[domanda_uppercase] = text

<think><set name="dialogue_act">setQuestion</set></think><think><set name="elemento1">josephine</set></think><think><set name="elemento2">none</set></think><think><set name="insieme">A</set></think><think><set name="topic">funzione</set></think>La funzione associa l'elemento josephine dell'insieme A all'elemento bari dell'insieme B.<image>insiemi/funzione.svg</image><svgElement style-name="stroke" style-value="#04ed00">link-josephine-bari</svgElement>
<think><set name="dialogue_act">setQuestion</set></think><think><set name="elemento1">silvia</set></think><think><set name="elemento2">none</set></think><think><set name="insieme">A</set></think><think><set name="topic">funzione</set></think>La funzione associa l'elemento silvia dell'insieme A all'elemento milano dell'insieme B.<image>insiemi/funzione.svg</image><svgElement style-name="stroke" style-value="#04ed00">link-silvia-milano</svgElement>
<think><set name="dialogue_act">setQuestion</set></think><think><set name="elemento1">luigi</

6. A cosa è collegato l'elemento del codominio

In [112]:
for elemento in link:
    domande = [
        '* ' + link[elemento] + ' *'
    ]

    # variabili
    vars = {
        'dialogue_act': 'setQuestion',
        'elemento1': link[elemento],
        'elemento2': 'none',
        'insieme': nome_insieme_codominio,
        'topic': 'funzione_inversa',
    }

    text = ''
    for var in vars:
        text += '<think><set name="' + var + '">' + vars[var] + '</set></think>'


    # contenuto testuale
    text += 'L\'elemento ' + link[elemento] + ' dell\'insieme ' + nome_insieme_codominio + ' è associato all\'elemento ' + elemento + ' dell\'insieme ' + nome_insieme_dominio + '.'
   
    # immagine
    text += '<image>insiemi/funzione.svg</image>'
    text += '<svgElement style-name="stroke" style-value="#04ed00">link-' + elemento + '-' + link[elemento] + '</svgElement>'

    print(text)

    for domanda in domande:
        domanda_uppercase = domanda.upper()
        setQuestionAnswer[domanda_uppercase] = text

<think><set name="dialogue_act">setQuestion</set></think><think><set name="elemento1">bari</set></think><think><set name="elemento2">none</set></think><think><set name="insieme">B</set></think><think><set name="topic">funzione_inversa</set></think>L'elemento bari dell'insieme B è associato all'elemento josephine dell'insieme A.<image>insiemi/funzione.svg</image><svgElement style-name="stroke" style-value="#04ed00">link-josephine-bari</svgElement>
<think><set name="dialogue_act">setQuestion</set></think><think><set name="elemento1">milano</set></think><think><set name="elemento2">none</set></think><think><set name="insieme">B</set></think><think><set name="topic">funzione_inversa</set></think>L'elemento milano dell'insieme B è associato all'elemento silvia dell'insieme A.<image>insiemi/funzione.svg</image><svgElement style-name="stroke" style-value="#04ed00">link-silvia-milano</svgElement>
<think><set name="dialogue_act">setQuestion</set></think><think><set name="elemento1">milano</set>

### Ta:request

Esempio:
- parlami del dominio
- descrivimi il codominio

1. Parlami del dominio

In [113]:
domande = [
    '* parlami * dominio *',
    '* parla * dominio *',
    '* parlami * ' + nome_insieme_dominio + ' *',
    '* parla * ' + nome_insieme_dominio + ' *',
    '* spiegami * dominio *',
    '* spiega * dominio *',
    '* spiegami * ' + nome_insieme_dominio + ' *',
    '* spiega * ' + nome_insieme_dominio + ' *',
    '* descrivimi * dominio *',
    '* descrivi * dominio *',
    '* descrivimi * ' + nome_insieme_dominio + ' *',
    '* descrivi * ' + nome_insieme_dominio + ' *',
    '* raccontami * dominio *',
    '* racconta * dominio *',
    '* raccontami * ' + nome_insieme_dominio + ' *',
    '* racconta * ' + nome_insieme_dominio + ' *',
]


# variabili
vars = {
    'dialogue_act': 'request',
    'elemento1': 'none',
    'elemento2': 'none',
    'insieme': nome_insieme_dominio,
    'topic': 'insieme',
}

text = ''
for var in vars:
    text += '<think><set name="' + var + '">' + nome_insieme_dominio + '</set></think>'

# contenuto testuale
text += 'L\'insieme ' + nome_insieme_dominio + ' è composto da ' + str(len(elementi_dominio)) + ' elementi: '
for elemento in elementi_dominio:
    text += elemento + ', '
text = text[:-2] + '.'

# immagine
text += '<image>insiemi/funzione.svg</image>'
text += '<svgElement style-name="fill" style-value="#04ed00">insieme-dominio</svgElement>'
print(text)

for domanda in domande:
    domanda_uppercase = domanda.upper()
    requestAnswer[domanda_uppercase] = text

<think><set name="dialogue_act">A</set></think><think><set name="elemento1">A</set></think><think><set name="elemento2">A</set></think><think><set name="insieme">A</set></think><think><set name="topic">A</set></think>L'insieme A è composto da 4 elementi: josephine, kevin, luigi, silvia.<image>insiemi/funzione.svg</image><svgElement style-name="fill" style-value="#04ed00">insieme-dominio</svgElement>


2. Parlami del codominio

In [114]:
domande = [
    '* parlami * codominio *',
    '* parla * codominio *',
    '* parlami * ' + nome_insieme_codominio + ' *',
    '* parla * ' + nome_insieme_codominio + ' *',
    '* spiegami * codominio *',
    '* spiega * codominio *',
    '* spiegami * ' + nome_insieme_codominio + ' *',
    '* spiega * ' + nome_insieme_codominio + ' *',
    '* descrivimi * codominio *',
    '* descrivi * codominio *',
    '* descrivimi * ' + nome_insieme_codominio + ' *',
    '* descrivi * ' + nome_insieme_codominio + ' *',
    '* raccontami * codominio *',
    '* racconta * codominio *',
    '* raccontami * ' + nome_insieme_codominio + ' *',
    '* racconta * ' + nome_insieme_codominio + ' *',
]

vars = {
    'dialogue_act': 'request',
    'elemento1': 'none',
    'elemento2': 'none',
    'insieme': nome_insieme_codominio,
    'topic': 'insieme',
}

text = ''
for var in vars:
    text += '<think><set name="' + var + '">' + nome_insieme_codominio + '</set></think>'

# contenuto testuale
text += 'L\'insieme ' + nome_insieme_codominio + ' è composto da ' + str(len(elementi_codominio)) + ' elementi: '
for elemento in elementi_codominio:
    text += elemento + ', '
text = text[:-2] + '.'

# immagine
text += '<image>insiemi/funzione.svg</image>'
text += '<svgElement style-name="fill" style-value="#04ed00">insieme-codominio</svgElement>'
print(text)

for domanda in domande:
    domanda_uppercase = domanda.upper()
    requestAnswer[domanda_uppercase] = text

<think><set name="dialogue_act">B</set></think><think><set name="elemento1">B</set></think><think><set name="elemento2">B</set></think><think><set name="insieme">B</set></think><think><set name="topic">B</set></think>L'insieme B è composto da 5 elementi: bari, roma, milano, bologna, siena.<image>insiemi/funzione.svg</image><svgElement style-name="fill" style-value="#04ed00">insieme-codominio</svgElement>


3. Parlami degli insiemi

In [115]:
domande = [
    '* parlami * dominio * codominio *',
    '* parla * dominio * codominio *',
    '* parlami * codominio * dominio *',
    '* parla * codominio * dominio *',
    '* parlami * ' + nome_insieme_codominio + ' * ' + nome_insieme_dominio + ' *',
    '* parla * ' + nome_insieme_codominio + ' * ' + nome_insieme_dominio + ' *',
    '* parlami * ' + nome_insieme_dominio + ' * ' + nome_insieme_codominio + ' *',
    '* parla * ' + nome_insieme_dominio + ' * ' + nome_insieme_codominio + ' *',
    '* parlami * insiemi *',
    '* parla * insiemi *',
    '* spiagami * dominio * codominio *',
    '* spiega * dominio * codominio *',
    '* spiegami * codominio * dominio *',
    '* spiega * codominio * dominio *',
    '* spiegami * ' + nome_insieme_codominio + ' * ' + nome_insieme_dominio + ' *',
    '* spiega * ' + nome_insieme_codominio + ' * ' + nome_insieme_dominio + ' *',
    '* spiegami * ' + nome_insieme_dominio + ' * ' + nome_insieme_codominio + ' *',
    '* spiega * ' + nome_insieme_dominio + ' * ' + nome_insieme_codominio + ' *',
    '* spiegami * insiemi *',
    '* spiega * insiemi *',
    '* descrivimi * dominio * codominio *',
    '* descrivi * dominio * codominio *',
    '* descrivimi * codominio * dominio *',
    '* descrivi * codominio * dominio *',
    '* descrivimi * ' + nome_insieme_codominio + ' * ' + nome_insieme_dominio + ' *',
    '* descrivi * ' + nome_insieme_codominio + ' * ' + nome_insieme_dominio + ' *',
    '* descrivimi * ' + nome_insieme_dominio + ' * ' + nome_insieme_codominio + ' *',
    '* descrivi * ' + nome_insieme_dominio + ' * ' + nome_insieme_codominio + ' *',
    '* descrivimi * insiemi *',
    '* descrivi * insiemi *',
    '* raccontami * dominio * codominio *',
    '* racconta * dominio * codominio *',
    '* raccontami * codominio * dominio *',
    '* racconta * codominio * dominio *',
    '* raccontami * ' + nome_insieme_codominio + ' * ' + nome_insieme_dominio + ' *',
    '* racconta * ' + nome_insieme_codominio + ' * ' + nome_insieme_dominio + ' *',
    '* raccontami * ' + nome_insieme_dominio + ' * ' + nome_insieme_codominio + ' *',
    '* racconta * ' + nome_insieme_dominio + ' * ' + nome_insieme_codominio + ' *',
    '* raccontami * insiemi *',
    '* racconta * insiemi *',
]

# variabili
vars = {
    'dialogue_act': 'request',
    'elemento1': 'none',
    'elemento2': 'none',
    'insieme': nome_insieme_dominio + '-' + nome_insieme_codominio,
    'topic': 'insieme',
}

text = ''
for var in vars:
    text += '<think><set name="' + var + '">' + vars[var] + '</set></think>'

# contenuto testuale
text += 'Gli insiemi nel nostro esempio sono 2. Il primo insieme, l\'insieme ' + nome_insieme_dominio + ', detto in questo '
text += 'caso dominio, è composto da ' + str(len(elementi_dominio)) + ' elementi: '
for elemento in elementi_dominio:
    text += elemento + ', '
text = text[:-2] + '.'

text += ' Il secondo insieme, l\'insieme ' + nome_insieme_codominio + ', noto come codominio, è invece composto '
text += 'da ' + str(len(elementi_codominio)) + ' elementi: '
for elemento in elementi_codominio:
    text += elemento + ', '
text = text[:-2] + '.'


text += '<image>insiemi/funzione.svg</image>'
text += '<svgElement style-name="fill" style-value="#04ed00">insieme-codominio</svgElement>'
text += '<svgElement style-name="fill" style-value="#04ed00">insieme-dominio</svgElement>'
print(text)

for domanda in domande:
    domanda_uppercase = domanda.upper()
    requestAnswer[domanda_uppercase] = text

<think><set name="dialogue_act">request</set></think><think><set name="elemento1">none</set></think><think><set name="elemento2">none</set></think><think><set name="insieme">A-B</set></think><think><set name="topic">insieme</set></think>Gli insiemi nel nostro esempio sono 2. Il primo insieme, l'insieme A, detto in questo caso dominio, è composto da 4 elementi: josephine, kevin, luigi, silvia. Il secondo insieme, l'insieme B, noto come codominio, è invece composto da 5 elementi: bari, roma, milano, bologna, siena.<image>insiemi/funzione.svg</image><svgElement style-name="fill" style-value="#04ed00">insieme-codominio</svgElement><svgElement style-name="fill" style-value="#04ed00">insieme-dominio</svgElement>


4. Descrivi la funzione

In [116]:
domande = [
    '* descrivi * funzione *',
    '* descrivimi * funzione *',
    '* descrivi * f *',
    '* descrivimi * f *',
    '* raccontami * funzione *',
    '* racconta * funzione *',
    '* raccontami * f *',
    '* racconta * f *',
    '* spiegami * funzione *',
    '* spiega * funzione *',
    '* spiegami * f *',
    '* spiega * f *',
    '* parlami * funzione *',
    '* parla * funzione *',
    '* parlami * f *',
    '* parla * f *',
]

# variabili
vars = {
    'dialogue_act': 'request',
    'elemento1': 'none',
    'elemento2': 'none',
    'insieme': 'none',
    'topic': 'funzione',
}

text = ''
for var in vars:
    text += '<think><set name="' + var + '">' + vars[var] + '</set></think>'

# contenuto testuale
text += 'La funzione associa gli elementi dell\'insieme ' + nome_insieme_dominio + ' con gli elementi dell\'insieme '
text += nome_insieme_codominio + '. In particolare, '

for componente in link:
    text += 'l\'elemento ' + componente + ' dell\'insieme ' + nome_insieme_dominio
    text += ' è associato all\'elemento ' + link[componente] + ' dell\'insieme ' + nome_insieme_codominio + ', '
text = text[:-2] + '.'

# immagine
text += '<image>insiemi/funzione.svg</image>'
for componente in link:
    text += '<svgElement style-name="stroke" style-value="#04ed00">link-' + componente + '-' + link[componente] + '</svgElement>'

for domanda in domande:
    domanda_uppercase = domanda.upper()
    requestAnswer[domanda_uppercase] = text

### Ta:propositionalQuestion

Esempio:
- L'elemento a è collegato all'elemento b?
- Esiste c nel dominio?
- ...

1. Elemento a è associto all'elemento b?

In [117]:
for elemento in elementi_dominio:
    for elemento2 in elementi_codominio:
        domande = [
            '* ' + elemento + ' * associato * ' + elemento2 + ' *',
            '* ' + elemento + ' * collegato * ' + elemento2 + ' *',
            '* funzione * ' + elemento + ' * ' + elemento2 + ' *',
            '* f * ' + elemento + ' * ' + elemento2 + ' *',
        ]

        # variabili
        vars = {
            'dialogue_act': 'propositionalQuestion',
            'elemento1': elemento,
            'elemento2': elemento2,
            'insieme': 'none',
            'topic': 'funzione',
        }

        text = ''
        for var in vars:
            text += '<think><set name="' + var + '">' + vars[var] + '</set></think>'

        # contenuto testuale
        if elemento2 == link[elemento]:
            text += elemento + ' è immagine di ' + elemento2 + '.'
        else:
            text += elemento + ' non è immagine di ' + elemento2 + '.'

        # immagine
        if elemento2 == link[elemento]:
            text += '<image>insiemi/funzione.svg</image>'
            text += '<svgElement style-name="stroke" style-value="#04ed00">link-' + elemento + '-' + elemento2 + '</svgElement>'
        else:
            text += '<image>insiemi/funzione.svg</image>'
            text += '<svgElement style-name="stroke" style-value="#04ed00">' + elemento + '</svgElement>'

        for domanda in domande:
            domanda_uppercase = domanda.upper()
            propositionalQuestionAnswer[domanda_uppercase] = text
            print(domanda, text)

* josephine * associato * bari * <think><set name="dialogue_act">propositionalQuestion</set></think><think><set name="elemento1">josephine</set></think><think><set name="elemento2">bari</set></think><think><set name="insieme">none</set></think><think><set name="topic">funzione</set></think>josephine è immagine di bari.<image>insiemi/funzione.svg</image><svgElement style-name="stroke" style-value="#04ed00">link-josephine-bari</svgElement>
* josephine * collegato * bari * <think><set name="dialogue_act">propositionalQuestion</set></think><think><set name="elemento1">josephine</set></think><think><set name="elemento2">bari</set></think><think><set name="insieme">none</set></think><think><set name="topic">funzione</set></think>josephine è immagine di bari.<image>insiemi/funzione.svg</image><svgElement style-name="stroke" style-value="#04ed00">link-josephine-bari</svgElement>
* funzione * josephine * bari * <think><set name="dialogue_act">propositionalQuestion</set></think><think><set name=

2. Elemento a è nel dominio? SI

In [118]:
for elemento in elementi_dominio:
    domande = [
        '* ' + elemento + ' * insieme ' + nome_insieme_dominio + ' *',
        '* ' + elemento + ' * dominio *',
    ]

    # variabili
    vars = {
        'dialogue_act': 'propositionalQuestion',
        'elemento1': elemento,
        'elemento2': 'none',
        'insieme': nome_insieme_dominio,
        'topic': 'insieme',
    }

    text = ''
    for var in vars:
        text += '<think><set name="' + var + '">' + vars[var] + '</set></think>'

    # contenuto testuale
    text += '' + elemento + ' è nell\'insieme ' + nome_insieme_dominio + '.'

    # immagine
    text += '<image>insiemi/funzione.svg</image>'
    text += '<svgElement style-name="stroke" style-value="#04ed00">' + elemento + '</svgElement>'

    for domanda in domande:
        domanda_uppercase = domanda.upper()
        propositionalQuestionAnswer[domanda_uppercase] = text
        print(domanda, text)

* josephine * insieme A * <think><set name="dialogue_act">propositionalQuestion</set></think><think><set name="elemento1">josephine</set></think><think><set name="elemento2">none</set></think><think><set name="insieme">A</set></think><think><set name="topic">insieme</set></think>josephine è nell'insieme A.<image>insiemi/funzione.svg</image><svgElement style-name="stroke" style-value="#04ed00">josephine</svgElement>
* josephine * dominio * <think><set name="dialogue_act">propositionalQuestion</set></think><think><set name="elemento1">josephine</set></think><think><set name="elemento2">none</set></think><think><set name="insieme">A</set></think><think><set name="topic">insieme</set></think>josephine è nell'insieme A.<image>insiemi/funzione.svg</image><svgElement style-name="stroke" style-value="#04ed00">josephine</svgElement>
* kevin * insieme A * <think><set name="dialogue_act">propositionalQuestion</set></think><think><set name="elemento1">kevin</set></think><think><set name="elemento2

3. Elemento a nel dominio? NO

In [119]:
for elemento in elementi_codominio:
    domande = [
        '* ' + elemento + ' * insieme ' + nome_insieme_dominio + ' *',
        '* ' + elemento + ' * dominio *',
    ]

    # variabili
    vars = {
        'dialogue_act': 'propositionalQuestion',
        'elemento1': elemento,
        'elemento2': 'none',
        'insieme': nome_insieme_codominio,
        'topic': 'insieme',
    }

    text = ''
    for var in vars:
        text += '<think><set name="' + var + '">' + vars[var] + '</set></think>'

    # contenuto testuale
    text += elemento + ' non è nell\'insieme ' + nome_insieme_dominio + '.'
    
    # immagine
    text += '<image>insiemi/funzione.svg</image>'
    text += '<svgElement style-name="stroke" style-value="#04ed00">' + elemento + '</svgElement>'

    for domanda in domande:
        domanda_uppercase = domanda.upper()
        propositionalQuestionAnswer[domanda_uppercase] = text
        print(domanda, text)

* bari * insieme A * <think><set name="dialogue_act">propositionalQuestion</set></think><think><set name="elemento1">bari</set></think><think><set name="elemento2">none</set></think><think><set name="insieme">B</set></think><think><set name="topic">insieme</set></think>bari non è nell'insieme A.<image>insiemi/funzione.svg</image><svgElement style-name="stroke" style-value="#04ed00">bari</svgElement>
* bari * dominio * <think><set name="dialogue_act">propositionalQuestion</set></think><think><set name="elemento1">bari</set></think><think><set name="elemento2">none</set></think><think><set name="insieme">B</set></think><think><set name="topic">insieme</set></think>bari non è nell'insieme A.<image>insiemi/funzione.svg</image><svgElement style-name="stroke" style-value="#04ed00">bari</svgElement>
* roma * insieme A * <think><set name="dialogue_act">propositionalQuestion</set></think><think><set name="elemento1">roma</set></think><think><set name="elemento2">none</set></think><think><set na

4. Elemento a nel codominio? SI

In [120]:
for elemento in elementi_codominio:
    domande = [
        '* ' + elemento + ' * insieme ' + nome_insieme_codominio + ' *',
        '* ' + elemento + ' * codominio *',
    ]

    # variabili
    vars = {
        'dialogue_act': 'propositionalQuestion',
        'elemento1': elemento,
        'elemento2': 'none',
        'insieme': nome_insieme_codominio,
        'topic': 'insieme',
    }

    text = ''
    for var in vars:
        text += '<think><set name="' + var + '">' + vars[var] + '</set></think>'

    # contenuto testuale
    text += elemento + ' è nell\'insieme ' + nome_insieme_codominio + '.'

    # immagine
    text += '<image>insiemi/funzione.svg</image>'
    text += '<svgElement style-name="stroke" style-value="#04ed00">' + elemento + '</svgElement>'
    
    for domanda in domande:
        domanda_uppercase = domanda.upper()
        propositionalQuestionAnswer[domanda_uppercase] = text
        print(domanda, text)   

* bari * insieme B * <think><set name="dialogue_act">propositionalQuestion</set></think><think><set name="elemento1">bari</set></think><think><set name="elemento2">none</set></think><think><set name="insieme">B</set></think><think><set name="topic">insieme</set></think>bari è nell'insieme B.<image>insiemi/funzione.svg</image><svgElement style-name="stroke" style-value="#04ed00">bari</svgElement>
* bari * codominio * <think><set name="dialogue_act">propositionalQuestion</set></think><think><set name="elemento1">bari</set></think><think><set name="elemento2">none</set></think><think><set name="insieme">B</set></think><think><set name="topic">insieme</set></think>bari è nell'insieme B.<image>insiemi/funzione.svg</image><svgElement style-name="stroke" style-value="#04ed00">bari</svgElement>
* roma * insieme B * <think><set name="dialogue_act">propositionalQuestion</set></think><think><set name="elemento1">roma</set></think><think><set name="elemento2">none</set></think><think><set name="in

5. Elemento a nel codominio? NO

In [121]:
for elemento in elementi_dominio:
    domande = [
        '* ' + elemento + ' * insieme ' + nome_insieme_codominio + ' *',
        '* ' + elemento + ' * codominio *',
    ]

    # variabili
    vars = {
        'dialogue_act': 'propositionalQuestion',
        'elemento1': elemento,
        'elemento2': 'none',
        'insieme': nome_insieme_codominio,
        'topic': 'insieme',
    }

    text = ''
    for var in vars:
        text += '<think><set name="' + var + '">' + vars[var] + '</set></think>'

    # contenuto testuale
    text += elemento + ' non è nell\'insieme ' + nome_insieme_codominio + '.'

    # immagine
    text += '<image>insiemi/funzione.svg</image>'
    text += '<svgElement style-name="stroke" style-value="#04ed00">' + elemento + '</svgElement>'

    for domanda in domande:
        domanda_uppercase = domanda.upper()
        propositionalQuestionAnswer[domanda_uppercase] = text
        print(domanda, text)

* josephine * insieme B * <think><set name="dialogue_act">propositionalQuestion</set></think><think><set name="elemento1">josephine</set></think><think><set name="elemento2">none</set></think><think><set name="insieme">B</set></think><think><set name="topic">insieme</set></think>josephine non è nell'insieme B.<image>insiemi/funzione.svg</image><svgElement style-name="stroke" style-value="#04ed00">josephine</svgElement>
* josephine * codominio * <think><set name="dialogue_act">propositionalQuestion</set></think><think><set name="elemento1">josephine</set></think><think><set name="elemento2">none</set></think><think><set name="insieme">B</set></think><think><set name="topic">insieme</set></think>josephine non è nell'insieme B.<image>insiemi/funzione.svg</image><svgElement style-name="stroke" style-value="#04ed00">josephine</svgElement>
* kevin * insieme B * <think><set name="dialogue_act">propositionalQuestion</set></think><think><set name="elemento1">kevin</set></think><think><set name=

## Costruzione dei file aiml

In [122]:
import xml.dom.minidom
import webbrowser

In [123]:
dialogue_acts = [
    'setQuestionAnswer',
    'requestAnswer',
    'propositionalQuestionAnswer',
]

In [124]:
for dialogue_act in dialogue_acts:
    file_path = 'funzione_' + dialogue_act + '.aiml'
    root = ET.Element('aiml')

    if dialogue_act == 'setQuestionAnswer':
        risposte = setQuestionAnswer
    elif dialogue_act == 'requestAnswer':
        risposte = requestAnswer
    elif dialogue_act == 'propositionalQuestionAnswer':
        risposte = propositionalQuestionAnswer
    else:
        break

    for domanda in risposte:
        #crea un tag xml chiamato category
        category = ET.Element('category')
        #inserisci all'interno un altro tag chiaamto pattern contentene un * e crea un tag chiamato template con il valore di text[0]
        pattern = ET.SubElement(category, 'pattern')
        pattern.text = domanda
        template = ET.SubElement(category, 'template')
        template.text = risposte[domanda]
        

        #aggiungi il tag category al tag root
        root.append(category)

    #salva il file xml
    tree = ET.ElementTree(root)
    tree.write(file_path)

    # Ottieni la rappresentazione del testo non escapato
    xml_str = xml.dom.minidom.parseString(ET.tostring(root)).toprettyxml(indent="    ")

    # Sovrascrivi il file AIML con le modifiche
    with open(file_path, 'w', encoding='utf-8') as file:
        file.write(xml_str)

    # Apri il file appena creato
    webbrowser.open(file_path)

    # Leggi il contenuto del file
    with open(file_path, 'r', encoding='utf-8') as file:
        file_content = file.read()

    # Sostituisci "&lt;" con "<" e "&gt;" con ">"
    file_content = file_content.replace("&lt;", "<").replace("&gt;", ">").replace("&quot;", "\"")

    # Sovrascrivi il file con le modifiche
    with open(file_path, 'w', encoding='utf-8') as file:
        file.write(file_content)